## Customer service automation (multi-agent):

## Agents:
### 1- Query Agent:Handles initial customer inquiries.
### 2- Resolution Agent:Solves technical or logistical issues.
### 3- Escalation Agent:Addresses complex queries oe escalates case to human representatives.

In [ ]:
from functools import partial
import operator
from typing import Annotated, Sequence, TypedDict, Literal
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_experimental.tools import PythonAstREPLTool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, BaseMessage
from pydantic import BaseModel
from langgraph.graph import END, StateGraph, START
from langgraph.prebuilt import create_react_agent


In [ ]:
# Define RouteResponse for customer service supervisor
class RouteResponseCS(BaseModel):
    next: Literal["Query_Agent", "Resolution_Agent", "Escalation_Agent", "FINISH"]

# Setup for customer service supervisor
members_cs = ["Query_Agent", "Resolution_Agent", "Escalation_Agent"]    
system_prompt_cs = f"You are a customer service supervisor managing agents:{','.join(members_cs)}."

# Create prompt template for the supervisor with correctly formatted options
prompt_cs = ChatPromptTemplate.from_template([
    ("system", system_prompt_cs),
    MessagesPlaceholder(variable_name="messages"),
    ("system", "Choose the next agent to act from {options}")
]).partial(options=str(members_cs))

# Define llm and supervisor function
llm = ChatOpenAI(model="gpt-4o-mini")

def supervisor_agent_cs(state):
    supervisor_chain_cs = prompt_cs | llm.with_structured_output(RouteResponseCS)
    return supervisor_chain_cs.invoke(state)

# Agent node function to handle message flow to each agent
def agent_node(state, agent, name):
    result = agent.invoke(state)
    return {"messages": [HumanMessage(content=result["messages"][-1].content, neme=name)]}

# Define agents for customer service task with realistic tools
query_agent = create_react_agent(llm, tools=[TavilySearchResults(max_results=5)])
resolution_agent = create_react_agent(llm, tools=[PythonAstREPLTool()])
escalation_agent = create_react_agent(llm, tools=[PythonAstREPLTool()])

# Create nodes for each agent with valid names
query_node = partial(agent_node, agent=query_agent,name="Query_Agent")
resolution_node = partial(agent_node, agent=resolution_agent, name="Resolution_Agent")
escalation_node = partial(agent_node, agent=escalation_agent, name="Escalation_Agent")

# Define customer service graph state and workflow
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    next: str

# Initialize StateGraph and add nodes
workflow = StateGraph(AgentState)
workflow.add_node("Query_Agent", query_node)
workflow.add_node("Resolution_Agent", resolution_node)
workflow.add_node("Escalation_Agent", escalation_node)
workflow.add_node("supervisor", supervisor_agent_cs)

# Define edges for agents to return to the supervisor
for member in members_cs:
    workflow.add_edge(member, "supervisor")

# Define conditional map for routing
conditional_map_cs = {k: k for k in members_cs}
conditional_map_cs["FINISH"] = END
workflow.add_conditional_edges("supervisor", lambda x: x["next"], conditional_map_cs)
workflow.add_edge(START, "supervisor")

# Compile and test for testing
graph_cs = workflow.compile()




In [ ]:
# Example input for testing
inputs = {"messages": [HumanMessage(content="Help me reset my password.")]}

# Run the graph
for output in graph_cs.stream(inputs):
    if "___end__" not in output:
        print(output)
        